In [1]:
import pandas as pd
import os
import glob
from IPython.display import display

# ==========================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN THEO CẤU TRÚC FOLDER MỚI
# ==========================================================
# Thư mục gốc của project
PROJECT_ROOT = r'D:\Machine Learning\ML-Lab1-Restaurant'

# Thư mục chứa các file nhãn thô
LABEL_FOLDER = os.path.join(PROJECT_ROOT, 'label')

# File gốc lấy từ thư mục preprocessed
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT, 'dataset', 'preprocessed', 'reviews_final_merged.csv')

# File sạch xuất ra thư mục model data
OUTPUT_FOLDER = os.path.join(PROJECT_ROOT, 'dataset', 'model data')
OUTPUT_FINAL_CLEAN = os.path.join(OUTPUT_FOLDER, 'final_dataset_cleaned.csv')

# Tạo thư mục đầu ra nếu chưa tồn tại
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"-> Đã tạo thư mục mới: {OUTPUT_FOLDER}")

# ==========================================================
# 2. GỘP NHÃN (TỰ ĐỘNG XỬ LÝ SỐ CỘT)
# ==========================================================
print("--- 1. ĐANG THU THẬP DỮ LIỆU NHÃN TỪ THƯ MỤC LABEL ---")
all_files = glob.glob(os.path.join(LABEL_FOLDER, "*.csv"))

# Danh sách các file cần loại trừ (nếu lỡ tay để trong folder label)
exclude_names = ['reviews_final_merged.csv', 'final_dataset_cleaned.csv', 'labels_merged_answer.csv']

list_dfs = []
for f in all_files:
    fname = os.path.basename(f)
    if fname in exclude_names: continue
    
    print(f"   + Đang đọc: {fname}")
    df_temp = pd.read_csv(f, header=None)
    
    # Lấy cột đầu (ID) và cột cuối (Label), bỏ qua các cột review ở giữa
    if df_temp.shape[1] >= 2:
        df_temp = df_temp.iloc[:, [0, -1]]
    
    # Xử lý xóa dòng header nếu có
    if not str(df_temp.iloc[0, 0]).isdigit():
        df_temp = df_temp.drop(df_temp.index[0])
        
    df_temp.columns = ['id', 'label']
    list_dfs.append(df_temp)

# Gộp và làm sạch kiểu dữ liệu
df_raw = pd.concat(list_dfs, ignore_index=True)
df_raw['id'] = pd.to_numeric(df_raw['id'], errors='coerce')
df_raw['label'] = pd.to_numeric(df_raw['label'], errors='coerce')
df_raw = df_raw.dropna(subset=['id', 'label'])
df_raw['id'] = df_raw['id'].astype(int)

# Xóa trùng ID (giữ dòng đầu tiên)
df_label_clean = df_raw.drop_duplicates(subset=['id'], keep='first')

# ==========================================================
# 3. KHỚP VỚI FILE TỪ PREPROCESSED VÀ KIỂM TRA
# ==========================================================
print(f"\n--- 2. ĐANG ĐỐI SOÁT VỚI FILE GỐC TẠI PREPROCESSED ---")
df_orig = pd.read_csv(ORIGINAL_DATA_FILE)
id_col = df_orig.columns[0]
df_orig[id_col] = df_orig[id_col].astype(int)

orig_ids = set(df_orig[id_col])
label_ids = set(df_label_clean['id'])

missing = orig_ids - label_ids
print(f"- Tổng dòng trong file preprocessed: {len(df_orig)}")
print(f"- Tổng số nhãn thu thập được: {len(df_label_clean)}")

if not missing:
    print("✅ KẾT QUẢ: Đã gán nhãn đầy đủ cho tất cả review.")
else:
    print(f"⚠️ CẢNH BÁO: Còn thiếu {len(missing)} ID chưa gán nhãn!")

# ==========================================================
# 4. MERGE VÀ XUẤT RA MODEL DATA
# ==========================================================
df_label_clean.columns = [id_col, 'label']
df_merged = pd.merge(df_orig, df_label_clean, on=id_col, how='inner')

# Loại bỏ nhãn 3 (tiếng nước ngoài)
df_final = df_merged[df_merged['label'] != 3].copy()

# Định dạng ID về dạng 5 chữ số (00001)
df_final[id_col] = df_final[id_col].apply(lambda x: f"{x:05d}")

# Lưu kết quả cuối cùng
df_final.to_csv(OUTPUT_FINAL_CLEAN, index=False, encoding='utf-8-sig')

print(f"\n--- HOÀN TẤT ---")
print(f"📂 Nguồn review: dataset/preprocessed/{os.path.basename(ORIGINAL_DATA_FILE)}")
print(f"🚀 File đích: dataset/model data/{os.path.basename(OUTPUT_FINAL_CLEAN)}")
print(f"✅ Tổng số mẫu tiếng Việt sẵn sàng để train: {len(df_final)}")

d:\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


--- 1. ĐANG THU THẬP DỮ LIỆU NHÃN TỪ THƯ MỤC LABEL ---
   + Đang đọc: 00001-08000.csv
   + Đang đọc: 08001_16000.csv
   + Đang đọc: 16001-24000.csv
   + Đang đọc: 24001-32000.csv
   + Đang đọc: 32001_32807.csv

--- 2. ĐANG ĐỐI SOÁT VỚI FILE GỐC TẠI PREPROCESSED ---
- Tổng dòng trong file preprocessed: 32807
- Tổng số nhãn thu thập được: 32807
✅ KẾT QUẢ: Đã gán nhãn đầy đủ cho tất cả review.

--- HOÀN TẤT ---
📂 Nguồn review: dataset/preprocessed/reviews_final_merged.csv
🚀 File đích: dataset/model data/final_dataset_cleaned.csv
✅ Tổng số mẫu tiếng Việt sẵn sàng để train: 32713
